# KIỂM THỬ VÀ PHÂN TÍCH DỮ LIỆU VECTOR HNSW TRÊN DỮ LIỆU THẬT
(Ứng dụng tinh hoa Big Data: Dask, Polars, DuckDB, Datashader, Panel)

**Mục tiêu:** Áp dụng toàn bộ kiến thức từ chuỗi 12 bài mẫu trực tiếp vào phân tích **dữ liệu thực tế** của dự án (16.45 triệu bản ghi). Để vừa sức với môi trường Jupyter cục bộ, ta sẽ trích xuất và tương tác với mẫu 50,000 vector gốc từ `vectors_int8.dat` và phân tích trên `shard_00000.jsonl` (1.06 GB).

In [1]:
import os
import time
import json
import numpy as np
import pandas as pd
import dask.dataframe as dd
import polars as pl
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import hvplot.pandas
import hvplot.dask
import datashader as ds
import datashader.transfer_functions as tf
import panel as pn

pn.extension()
sns.set_theme(style='whitegrid')
os.makedirs('assets/figs', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# Đường dẫn file dữ liệu thật của dự án
JSONL_SHARD = 'data/crawl/shard_00000.jsonl'
VECTOR_BIN = 'data/quantized/vectors_int8.dat'
Q_PARAMS = 'data/quantized/quantization_params.json'

## 1. Dask & Định dạng Parquet (Bài 03, 04)
Đọc dữ liệu Crawl thô (1.06 GB JSONL) bằng Dask và chuyển sang dạng Parquet. Cấu trúc dạng cột (columnar) của Parquet sẽ giúp tăng tốc độ đọc dữ liệu hàng chục lần so với JSONL gốc.

In [2]:
print("Bắt đầu đọc JSONL bằng Polars...")
parquet_path = "data/processed/shard_00000_optimized.parquet"
if not os.path.exists(parquet_path):
    pl_df_raw = pl.read_ndjson(JSONL_SHARD)
    print(pl_df_raw.head())
    # Lọc cột và lưu xuống Parquet
    pl_df_subset = pl_df_raw.select(["doc_id", "title", "token_count", "crawled_at"])
    pl_df_subset.write_parquet(parquet_path)
    print("Đã lưu cache Parquet thành công.")
else:
    print("Đã tìm thấy Parquet tối ưu.")


Bắt đầu đọc JSONL bằng Polars...
Đã tìm thấy Parquet tối ưu.


## 2. Big Data Analysis với Polars và DuckDB (Bài 10, 11)
Thực thi các lệnh truy vấn phân tích (SQL) trên tập dữ liệu đã lưu bằng các công cụ tính toán siêu tốc (Out-of-core engine).

In [3]:
# 2.1 Polars: Xử lý DataFrame bằng lõi Rust
print("=== Polars Engine ===")
pl_df = pl.read_parquet(parquet_path)
print("Thống kê số lượng Token trung bình theo chiều dài:")
print(pl_df.select([
    pl.col("token_count").mean().alias("avg_tokens"),
    pl.col("token_count").max().alias("max_tokens"),
    pl.count().alias("total_docs")
]))

# 2.2 DuckDB: Phân tích trực tiếp SQL
print("\n=== DuckDB SQL Engine ===")
query = f"""
    SELECT 
        CAST(crawled_at AS DATE) as crawl_date,
        COUNT(*) as doc_count,
        AVG(token_count) as avg_token
    FROM '{parquet_path}'
    GROUP BY crawl_date
    ORDER BY doc_count DESC
    LIMIT 5
"""
print(duckdb.query(query).df())

=== Polars Engine ===
Thống kê số lượng Token trung bình theo chiều dài:
shape: (1, 3)
┌────────────┬────────────┬────────────┐
│ avg_tokens ┆ max_tokens ┆ total_docs │
│ ---        ┆ ---        ┆ ---        │
│ f64        ┆ i64        ┆ u32        │
╞════════════╪════════════╪════════════╡
│ 3441.83742 ┆ 191338     ┆ 50000      │
└────────────┴────────────┴────────────┘

=== DuckDB SQL Engine ===
  crawl_date  doc_count   avg_token
0 2026-09-04      50000  3441.83742


C:\Users\dhp01\AppData\Local\Temp\ipykernel_9016\3806673689.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_docs")


## 3. Trích xuất Vector Lượng tử (SQ8) & Giải mã (Dequantize)
Dự án có tổng cộng ~16.45 triệu vector 384-D, được lưu dưới dạng file nhị phân `int8` kích thước 6.3GB. Ta sẽ load 50,000 vector đầu tiên để đánh giá PCA và test độ chính xác.

In [4]:
num_samples = 5000
dim = 384

# Đọc cấu hình lượng tử hóa
with open(Q_PARAMS, 'r') as f:
    q_params = json.load(f)
min_vals = np.array(q_params['min_vals'], dtype=np.float32)
scales = np.array(q_params['scales'], dtype=np.float32)

# Nạp 50,000 vector bằng np.memmap (Lazy Load từ file 6.3GB)
# SQ8 trong dự án thường lưu dạng uint8 (0-255)
mapped_vectors = np.memmap(VECTOR_BIN, dtype=np.uint8, mode='r', shape=(16459486, dim))
sq8_vectors = mapped_vectors[:num_samples]

# Giải mã (Dequantize) về Float32 để chạy Standard HNSW làm Ground Truth
float_vectors = (sq8_vectors.astype(np.float32) * scales) + min_vals
float_vectors = float_vectors / np.linalg.norm(float_vectors, axis=1, keepdims=True)

print(f"Đã nạp và giải mã {num_samples} vector thành công.")

Đã nạp và giải mã 5000 vector thành công.


## 4. PCA Datashader (Bài 02, 06)
Trực quan hóa sự phân bổ của 50,000 văn bản thực tế trong không gian vector.

In [5]:
# Giảm chiều xuống 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(float_vectors)
df_coords = pd.DataFrame(coords, columns=['x', 'y'])

# Dựa vào chiều dài token_count để đổi màu giả lập (Semantic density)
# Ta lấy token_count từ 50000 dòng đầu tiên của dữ liệu Polars (vì chúng map 1-1)
tokens = pl_df.head(num_samples)['token_count'].to_numpy()
df_coords['token_length'] = np.where(tokens > 300, 'Long', np.where(tokens > 150, 'Medium', 'Short'))

plt.figure(figsize=(10,6))
sns.scatterplot(data=df_coords, x='x', y='y', hue='token_length', s=5, alpha=0.5, palette='Set2')
plt.title('2D PCA Clusters of Real Vectors (Categorized by Length)')
plt.savefig('assets/figs/pca_clusters.png', dpi=300)
plt.close()
print("Đã xuất hình phân cụm PCA.")

Đã xuất hình phân cụm PCA.


## 5. Benchmark Hệ thống HNSW Core
Thực hiện truy vấn trên Float32 (Tiêu chuẩn) vs Uint8 (Two-Tier SQ8) dùng vector thực tế.

In [6]:
import sys
sys.path.append('./src')
from src.ann_index.two_tier_hnsw import TwoTierQuantizedHNSW
from src.ann_index.hnsw import StandardHNSWIndex
from src.ann_index.flat import FlatIndex

# Lấy 100 vector ngẫu nhiên làm Query
queries = float_vectors[:100].copy()
train_emb = float_vectors[100:]  # 49,900 vector làm Index

flat_idx = FlatIndex('l2')
flat_idx.build(train_emb)

standard_idx = StandardHNSWIndex(m=16, ef_construction=100)
standard_idx.build(train_emb)

two_tier_idx = TwoTierQuantizedHNSW(m=16, ef_search=50)
two_tier_idx.build(train_emb)

def evaluate(index, name):
    latencies, recalls = [], []
    for q in queries:
        gt_d, gt_i = flat_idx.search(q, 10)
        t0 = time.perf_counter()
        d, i = index.search(q, 10)
        t1 = time.perf_counter()
        
        latencies.append((t1 - t0) * 1000)
        intersect = len(np.intersect1d(np.array(gt_i).flatten(), np.array(i).flatten()))
        recalls.append(intersect / 10.0)
        
    return {
        'Algorithm': name,
        'Latency (ms)': np.mean(latencies),
        'QPS': 1000 / np.mean(latencies),
        'Recall@10 (%)': np.mean(recalls) * 100
    }

res_std = evaluate(standard_idx, 'Standard HNSW (Float32)')
res_tt = evaluate(two_tier_idx, 'Two-Tier HNSW (SQ8)')
benchmark_df = pd.DataFrame([res_std, res_tt])
print(benchmark_df)

                 Algorithm  Latency (ms)         QPS  Recall@10 (%)
0  Standard HNSW (Float32)      7.043028  141.984385           47.8
1      Two-Tier HNSW (SQ8)      3.026616  330.402007           30.5


## 6. Dashboard Báo Cáo (Bài 07)

In [7]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=benchmark_df, x='Algorithm', y='QPS', hue='Algorithm', palette='magma', legend=False)
plt.title('Thông lượng (QPS)')
plt.subplot(1, 2, 2)
sns.barplot(data=benchmark_df, x='Algorithm', y='Latency (ms)', hue='Algorithm', palette='viridis', legend=False)
plt.title('Độ trễ (ms)')
plt.savefig('assets/figs/perf_latency_qps.png', dpi=300)
plt.close()
print("Đã lưu biểu đồ Benchmark.")

try:
    title = pn.pane.Markdown("# 🚀 Báo Cáo Phân Tích Dữ Liệu Vector (Thực Tế)", width=800)
    metrics = pn.widgets.DataFrame(benchmark_df, name='Kết quả thuật toán', width=600)
    dashboard = pn.Column(title, metrics, background='#f0f0f0')
    dashboard.save('assets/figs/dashboard.html')
    print("Đã xuất Dashboard HTML.")
except Exception as e:
    print("Dashboard Panel:", e)


Đã lưu biểu đồ Benchmark.
Dashboard Panel: Column.__init__() got an unexpected keyword argument 'background'
